# 03 — MCP Over HTTP

**What you'll learn**
- Why you eventually outgrow stdio.
- How MCP looks the same logically but ships as JSON-RPC over HTTP/SSE.
- What changes for the client when the server is remote.
- The risks that appear the moment your tools live on a URL.

## stdio vs HTTP at a glance

| | stdio | HTTP / streamable-http |
|---|---|---|
| Who starts the server? | The host, as a subprocess | A separate long-running service |
| Where does it run? | Same machine | Any machine reachable over the network |
| Authentication? | Inherits OS user; usually trusted | **You must add it.** Anyone with the URL can call |
| Best for | Desktop AI apps, dev loops | Multi-user products, teams, cloud agents |
| Discovery | One stdio handshake | An HTTP request to `/mcp` (or similar) |
| Latency | µs | network RTT |

Same protocol, different transport. The MCP spec defines both so the same tools can serve both audiences.

## Simulated HTTP transport

We keep using the `MiniMCPServer` from notebook 02 but wrap it in a client that pretends to send JSON-RPC frames over the wire. That makes the request/response shape explicit.

In [ ]:
from typing import Callable, Any

class MiniMCPServer:
    """Tiny MCP-style server used for teaching.

    Real MCP defines a JSON-RPC protocol on top of a transport (stdio or HTTP).
    This class keeps only the three ideas you actually need to internalize:
      1. tools are registered with a name + description + schema
      2. a client can list them
      3. a client can call one by name with arguments
    """

    def __init__(self, name: str) -> None:
        self.name = name
        self._tools: dict[str, Callable[..., Any]] = {}
        self._descriptions: dict[str, str] = {}

    def tool(self, description: str = ""):
        """Decorator that registers a function as an MCP tool."""
        def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
            self._tools[func.__name__] = func
            self._descriptions[func.__name__] = description or (func.__doc__ or "").strip()
            return func
        return decorator

    def list_tools(self) -> list[dict]:
        """Discovery: what can I do?"""
        return [{"name": n, "description": self._descriptions[n]} for n in self._tools]

    def call_tool(self, name: str, arguments: dict) -> Any:
        """Execution: do the thing."""
        if name not in self._tools:
            raise ValueError(f"Unknown tool: {name}")
        return self._tools[name](**arguments)

In [ ]:
# Fake in-memory CRM. In production this would be HubSpot, Salesforce, etc.
CONTACTS: dict = {}
TASKS: list = []
OSC_TEAM = [
    {"id": "osc_101", "name": "Ava OSC",  "last_assigned": 0},
    {"id": "osc_102", "name": "Ben OSC",  "last_assigned": 0},
    {"id": "osc_103", "name": "Cara OSC", "last_assigned": 0},
]
_assignment_counter = 0
print("Fake CRM ready. Contacts:", len(CONTACTS), "OSCs:", len(OSC_TEAM))

In [ ]:
server = MiniMCPServer("sales-tools-http")

@server.tool(description="Create a new contact in the CRM")
def create_contact(name: str, email: str) -> dict:
    contact_id = f"contact_{len(CONTACTS) + 1}"
    contact = {"id": contact_id, "name": name, "email": email, "owner_id": None}
    CONTACTS[contact_id] = contact
    return contact

@server.tool(description="Assign an OSC to a contact using round-robin")
def assign_osc(contact_id: str) -> dict:
    global _assignment_counter
    chosen = min(OSC_TEAM, key=lambda osc: osc["last_assigned"])
    _assignment_counter += 1
    chosen["last_assigned"] = _assignment_counter
    CONTACTS[contact_id]["owner_id"] = chosen["id"]
    return chosen

@server.tool(description="Create a follow-up task for a contact")
def create_followup_task(contact_id: str, note: str) -> dict:
    task = {"id": f"task_{len(TASKS) + 1}", "contact_id": contact_id,
            "note": note, "status": "open"}
    TASKS.append(task)
    return task

print("tools:", [t["name"] for t in server.list_tools()])

## A pretend HTTP client

In real MCP, a tool call is JSON-RPC like:

```json
{"jsonrpc": "2.0", "id": 1, "method": "tools/call",
 "params": {"name": "create_contact",
            "arguments": {"name": "John", "email": "j@x.com"}}}
```

We mimic that shape so the conceptual jump to a real HTTP client is tiny.

In [ ]:
import itertools

class MiniHTTPClient:
    """Simulates an MCP-over-HTTP client. Builds JSON-RPC envelopes; the in-process server unpacks them."""

    def __init__(self, server: MiniMCPServer, url: str):
        self._server = server
        self.url = url
        self._ids = itertools.count(1)

    def _send(self, method: str, params: dict) -> dict:
        envelope = {"jsonrpc": "2.0", "id": next(self._ids), "method": method, "params": params}
        print(f"POST {self.url} {envelope}")
        if method == "tools/list":
            result = self._server.list_tools()
        elif method == "tools/call":
            result = self._server.call_tool(params["name"], params["arguments"])
        else:
            raise ValueError(method)
        response = {"jsonrpc": "2.0", "id": envelope["id"], "result": result}
        return response

    def list_tools(self) -> list[dict]:
        return self._send("tools/list", {})["result"]

    def call_tool(self, name: str, arguments: dict):
        return self._send("tools/call", {"name": name, "arguments": arguments})["result"]

http_client = MiniHTTPClient(server, url="http://localhost:8000/mcp")
http_client.list_tools()

## The host doesn't care about the transport

This is the punchline. The host code is identical to notebook 02 — only the client object changed. Same `list_tools`, same `call_tool`. That's what a protocol buys you.

In [ ]:
contact = http_client.call_tool("create_contact",
                                {"name": "John Doe", "email": "john@example.com"})
owner = http_client.call_tool("assign_osc", {"contact_id": contact["id"]})
task = http_client.call_tool("create_followup_task",
                             {"contact_id": contact["id"],
                              "note": "Follow up within 24 hours"})
print({"contact": contact, "owner": owner, "task": task})

## Mini test

In [ ]:
assert http_client.list_tools()[0]["name"] == "create_contact"
assert contact["owner_id"] is None or contact["owner_id"].startswith("osc_")
assert task["status"] == "open"
print("ok")

## The real thing — FastMCP over HTTP

Save this as `sales_mcp_server_http.py` and run it. The server listens on `http://localhost:8000/mcp`. Any host on your network — Claude Desktop, a coworker's laptop, a cloud agent — can connect to that URL.

In [ ]:
REAL_HTTP_SERVER = r'''# sales_mcp_server_http.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("sales-tools")

@mcp.tool()
def create_contact(name: str, email: str) -> dict:
    """Create a new contact in the CRM."""
    return {"id": "contact_1", "name": name, "email": email}

# ... assign_osc, create_followup_task as before ...

if __name__ == "__main__":
    # streamable-http is the modern MCP HTTP transport (replaces older SSE-only mode).
    mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)
'''
print(REAL_HTTP_SERVER)

In [ ]:
REAL_HTTP_CLIENT = r'''# host_http_client.py
import asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    async with streamablehttp_client("http://localhost:8000/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print([t.name for t in tools.tools])
            r = await session.call_tool("create_contact",
                                        {"name": "John Doe", "email": "j@example.com"})
            print(r)

asyncio.run(main())
'''
print(REAL_HTTP_CLIENT)

## What just got dangerous

Once the server is on a URL, **anyone who can reach it can call your tools**. That includes:

- Anyone on your local network if you bound to `0.0.0.0` without a firewall.
- Any browser tab if you forgot CORS rules.
- Anyone who guesses the URL once the server is public.
- An LLM run by another agent that found your URL in a config file it was asked to read.

Three issues to keep in mind every single time you ship an HTTP MCP server:

1. **Auth** — bearer tokens, OAuth, mTLS. Notebook 04.
2. **Input validation** — never trust the arguments. Notebook 05.
3. **Destructive tools** — `delete_*` over a URL is one prompt-injection away from disaster. Notebook 06.

You have not "deployed an MCP server" until you've thought about all three.

## Key takeaway

HTTP lets your tools serve multiple hosts and multiple machines, but it turns "a Python function I wrote" into "a public-ish API". Treat it like any other public API: assume the world can call it, and design defenses accordingly. The next three notebooks build those defenses one layer at a time.